# 02: Weekly Seasonality Analysis
This notebook analyzes Wikipedia's usage patterns based on the day of the week, focusing on identifying the typical usage profile and the evolution of the Weekend-to-Weekday traffic ratio from 2015 to 2025.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import numpy as np

# Add src to path
sys.path.append(os.path.abspath('../../'))
from src.data_prep import clean_pageview_data, add_time_features

# Configuration
DATA_PATH = '../../data/raw/en_wiki_pageviews_daily.csv'
REPORT_DIR = '../../reports/'
os.makedirs(REPORT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
PURPLE_COLOR = '#800080'

## 1. Data Loading and Preparation

In [ ]:
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = clean_pageview_data(df)
df = add_time_features(df)

# Ensure correct weekday ordering for plots and analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=day_order, ordered=True)

print(f"Data loaded from {DATA_PATH}")
df.head()

## 2. Weekly Usage Profile
Goal: understand typical traffic by day of week.

In [ ]:
# Compute mean views for each day
weekly_profile = df.groupby('day_of_week')['views'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=weekly_profile, x='day_of_week', y='views', palette='viridis')
plt.title('Wikipedia Weekly Usage Profile: Mean Views by Day', fontsize=15)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Average Pageviews', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_2_weekly_profile_bars.png'))
plt.show()

## 3. Annual Weekend vs Weekday Analysis
Goal: measure behavioral difference over time.

In [ ]:
# Group by year and is_weekend to get mean views
annual_ratios = df.groupby(['year', 'is_weekend'])['views'].mean().unstack()
annual_ratios.columns = ['weekday_mean', 'weekend_mean']

# Calculate the ratio: Weekend / Weekday
annual_ratios['ratio'] = annual_ratios['weekend_mean'] / annual_ratios['weekday_mean']

print("Yearly Weekend vs Weekday Ratio Table:")
print(annual_ratios[['weekday_mean', 'weekend_mean', 'ratio']])

## 4. Main Visualization: Weekend-to-Weekday Ratio Trend

In [ ]:
plt.figure(figsize=(14, 8))

# Plot the ratio line
plt.plot(annual_ratios.index, annual_ratios['ratio'], marker='o', linestyle='-', color=PURPLE_COLOR, linewidth=2.5, markersize=8)

# Add horizontal reference line at 1.0
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.6, label='Equal Weekend/Weekday Traffic')

# Formatting
plt.title('Ratio of Weekend to Weekday Pageviews (2015-2025)', fontsize=18, pad=20)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Weekend / Weekday Mean Ratio', fontsize=14)
plt.ylim(0.94, 1.04) 
plt.grid(True, which='both', linestyle='-', alpha=0.3)
plt.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_2_weekend_weekday_ratio.png'), dpi=300)
plt.show()

## 5. Stability Check (Variance per Weekday)

In [ ]:
# Calculate standard deviation and Coefficient of Variation (CV) for daily views per weekday
stability = df.groupby('day_of_week')['views'].agg(['mean', 'std']).reset_index()
stability['cv'] = stability['std'] / stability['mean']

print("Stability Check: Variation per Weekday (CV = Std / Mean)")
print(stability)

## 6. Insights and Interpretation
Based on the analysis above:

1. **Is Wikipedia primarily a weekday utility?** 
   * Prior to 2020, yes. The ratio was consistently below 1.0, indicating higher activity on school/work days.

2. **Are weekends becoming more active over time?**
   * Yes. There is a clear upward trend in the ratio, crossing the 1.0 line during the pandemic and staying above it.

3. **Is usage behavior stable or evolving?**
   * Evolving. The shift from weekday-dominant to weekend-dominant suggests that Wikipedia usage is increasingly integrated into leisure activities or that remote/flexible work has shifted typical school-driven traffic spikes.